# road-damage-shift — Semaine 1 : baselines YOLO11s vs RT-DETR-l

Entraînement sur **Japon + Inde + Tchéquie**, sélection sur la validation source, évaluation sur le test source
et sur le test de chaque pays (dont les pays cibles jamais vus = point « 0 image » de la courbe Q1).
Tout est tracé dans MLflow.

**Réglages Kaggle avant de lancer**
1. *Accelerator* : GPU T4 (un seul GPU est utilisé, pour la reproductibilité).
2. *Internet* : ON (pip, clone du dépôt, téléchargement des poids pré-entraînés Ultralytics).
3. *Données* : par défaut (`DOWNLOAD = True`) le notebook télécharge RDD2022 depuis figshare (≈ 13 Go) et n'extrait que les pays utiles.
   Sinon, ajoute en *Input* un jeu qui garde la structure `<Pays>/train/annotations/xmls` + `train/images`.
4. *Secrets* (optionnel, pour un MLflow distant type DagsHub) : `MLFLOW_TRACKING_URI`, `MLFLOW_TRACKING_USERNAME`, `MLFLOW_TRACKING_PASSWORD`.
   Sans secret, MLflow écrit dans `/kaggle/working/mlflow.db` (téléchargeable en fin de notebook).

**Ordre conseillé** : `SMOKE = True` en session interactive (≈ quelques minutes, vérifie toute la chaîne),
puis `SMOKE = False` via *Save Version → Save & Run All* (tourne en arrière-plan, 12 h max), **un modèle par version**.

In [ ]:
# ---- Paramètres -------------------------------------------------------------
REPO_URL = "https://github.com/<TON_USER>/road-damage-shift.git"  # à remplacer
REPO_REF = "main"
REPO_DIR = "/tmp/road-damage-shift"
DOWNLOAD = True                   # True : figshare officiel ; False : dataset ajouté en Input
COUNTRIES = ["Japan", "India", "Czech", "United_States"]  # pays extraits ; ajouter "Norway" seulement si besoin (9,9 Go)
RDD_ROOT = "/kaggle/input"        # si DOWNLOAD = False : découverte automatique des pays sous ce dossier
DATA_OUT = "/tmp/rdd_yolo"        # disque éphémère : images en liens symboliques, labels réécrits
SMOKE = True                      # True : 1 epoch sur 5 % du train ; False : protocole complet
MODELS_TO_RUN = ["yolo11s"]       # puis ["rtdetr_l"] dans une 2e version (budget 12 h/session)
TRAIN_SEED = None                 # None = seed du projet (42) ; 1, 2... pour mesurer la variance d'entraînement.
                                  # Ne change QUE l'entraînement : découpages et tests restent identiques.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf {REPO_DIR} && git clone -q --depth 1 -b {REPO_REF} {REPO_URL} {REPO_DIR}
!pip install -q -e "{REPO_DIR}[train]" faster-coco-eval  # requis par Ultralytics pour save_json

In [ ]:
import json, os, platform, re, subprocess, sys, time
from pathlib import Path

sys.path.insert(0, f"{REPO_DIR}/src")  # l'install éditable n'est visible qu'après redémarrage du noyau
import mlflow, numpy as np, pandas as pd, torch, ultralytics, yaml
from ultralytics import RTDETR, YOLO, settings

CFG = yaml.safe_load(open(f"{REPO_DIR}/configs/default.yaml"))
TRAIN, CLASSES = CFG["train"], CFG["data"]["classes"]
SEED = CFG["project"]["seed"] if TRAIN_SEED is None else TRAIN_SEED  # seed d'entraînement


def run_name_for(key):
    return f"baseline_{key}" + ("" if TRAIN_SEED is None else f"_seed{TRAIN_SEED}") + ("_smoke" if SMOKE else "")
GIT_SHA = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "ultralytics": ultralytics.__version__,
            "mlflow": mlflow.__version__, "cuda": torch.version.cuda,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}
print(GIT_SHA[:10], VERSIONS)
assert torch.cuda.is_available(), "Active le GPU T4 dans les réglages du notebook"

## 0. Données brutes : RDD2022 depuis figshare (dépôt officiel, licence CC BY 4.0)
Le serveur S3 du challenge renvoie désormais 403. Le zip figshare (≈ 13 Go) est téléchargé, vérifié par MD5,
puis **seuls les pays de `COUNTRIES` sont extraits** (Norvège et Chine ne sont jamais décompressées).
Compter 10 à 25 min ; `wget -nv` n'affiche rien avant la fin du téléchargement. Relancer la cellule reprend
le téléchargement là où il s'est arrêté, et ne refait rien si l'extraction est déjà faite.

In [ ]:
import hashlib, json, shutil, tempfile, urllib.request, zipfile

FIGSHARE_ARTICLE = 21431547  # RDD2022 sur figshare (source officielle, licence CC BY 4.0)
ALL_COUNTRIES = ["Japan", "India", "Czech", "Norway", "United_States", "China_MotorBike", "China_Drone"]
RAW = Path("/tmp/rdd_raw")


def _norm(s):
    return s.lower().replace(" ", "_").replace("-", "_")


def country_of(name):
    """Pays RDD2022 nommé dans un chemin d'archive (dossier 'Japan' ou zip 'RDD2022_Japan.zip'), sinon None."""
    stems = [_norm(p).removesuffix(".zip") for p in Path(name).parts]
    for c in sorted(ALL_COUNTRIES, key=len, reverse=True):
        cn = _norm(c)
        if any(s == cn or s.endswith("_" + cn) for s in stems):
            return c
    return None


def extract_all(zip_path, dest):
    """Décompresse un zip puis, récursivement, les zips qu'il contenait ; supprime les zips."""
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    Path(zip_path).unlink()
    while inner := sorted(Path(dest).rglob("*.zip")):
        for zp in inner:
            extract_all(zp, zp.parent)


def extract_countries(zip_path, dest, keep):
    """N'extrait que les pays de `keep`, quelle que soit l'organisation de l'archive
    (dossiers par pays, zips par pays, ou zip global imbriqué). Les autres pays ne sont jamais décompressés."""
    with zipfile.ZipFile(zip_path) as z:
        for m in z.infolist():
            if m.is_dir():
                continue
            c = country_of(m.filename)
            is_zip = m.filename.lower().endswith(".zip")
            if c is None and is_zip:  # archive englobante (ex. RDD2022.zip) : on descend dedans
                tmp_dir = Path(tempfile.mkdtemp(dir=dest, prefix="_nested_"))
                extract_countries(Path(z.extract(m, tmp_dir)), dest, keep)
                shutil.rmtree(tmp_dir)
            elif c in keep and is_zip:  # zip d'un pays voulu
                extract_all(Path(z.extract(m, dest / c)), dest / c)
            elif c in keep:  # fichier dans un dossier de pays voulu
                z.extract(m, dest)


def md5sum(path, chunk=1 << 24):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while b := f.read(chunk):
            h.update(b)
    return h.hexdigest()


if DOWNLOAD:
    RAW.mkdir(parents=True, exist_ok=True)
    marker = RAW / ("done_" + "-".join(sorted(COUNTRIES)))
    if marker.exists():
        print("Déjà extrait :", sorted(COUNTRIES))
    else:
        meta = json.load(urllib.request.urlopen(f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE}", timeout=60))
        f = max(meta["files"], key=lambda x: x["size"])
        zp = Path("/tmp") / f["name"]
        print(f"Téléchargement {f['name']} ({f['size'] / 1e9:.2f} Go), licence {meta['license']['name']}...")
        t0 = time.time()
        !wget -c -nv -O {zp} {f['download_url']}
        print(f"  {time.time() - t0:.0f} s")
        expected = f.get("computed_md5") or f.get("supplied_md5")
        if expected:
            got = md5sum(zp)
            if got != expected:
                zp.unlink()
                raise RuntimeError(f"MD5 différent ({got} != {expected}) : zip supprimé, relancer la cellule")
            print("  MD5 vérifié :", got)
        t0 = time.time()
        extract_countries(zp, RAW, set(COUNTRIES))
        zp.unlink()
        marker.touch()
        print(f"  extraction {time.time() - t0:.0f} s")
    n_xml = {}
    for x in RAW.rglob("*.xml"):
        c = country_of(str(x.relative_to(RAW)))
        n_xml[c] = n_xml.get(c, 0) + 1
    print("XML par pays :", n_xml)
    RDD_ROOT = str(RAW)
    !du -sh {RDD_ROOT}
print("RDD_ROOT =", RDD_ROOT)

## 1. Conversion VOC → YOLO (split par pays, sous-échantillonnage, listes few-shot figées)
**Avant d'entraîner, regarde `unknown_labels` et `dropped_boxes`** : une étiquette inattendue (coquille) se corrige via
`data.label_aliases` dans la config, puis on relance. Envoie-moi ces deux tableaux après le premier passage.

In [ ]:
!python -m roadshift.data.convert --config {REPO_DIR}/configs/default.yaml --raw-root {RDD_ROOT} --out {DATA_OUT} --overwrite

DATA = Path(DATA_OUT)
STATS = json.load(open(DATA / "stats.json"))
DATASET_INFO = json.load(open(DATA / "dataset_info.json"))
per = pd.DataFrame({k: {"images": v["images"], "sans_degat": v["background_images"],
                        "redimensionnees": v["resized_images"], **v["boxes"]}
                    for k, v in STATS["per_country_split"].items()}).T
display(per)
print("Étiquettes hors classes :", json.dumps(STATS["unknown_labels"], indent=1))
print("Boîtes ignorées :", json.dumps(STATS["dropped_boxes"], indent=1))
print("Images sans XML apparié :", STATS["missing_images"])
print("manifest sha256 :", DATASET_INFO["manifest_sha256"])

## 2. MLflow

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for k in ("MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if (v := _secret(k)):
        os.environ[k] = v
TRACKING_URI = CFG["mlflow"]["tracking_uri"] or _secret("MLFLOW_TRACKING_URI") or "sqlite:////kaggle/working/mlflow.db"
os.chdir("/kaggle/working")  # artefacts locaux -> /kaggle/working/mlruns
mlflow.set_tracking_uri(TRACKING_URI)
EXPERIMENT = CFG["mlflow"]["experiment"] + ("-smoke" if SMOKE else "")
mlflow.set_experiment(EXPERIMENT)
settings.update({"mlflow": False})  # on journalise nous-mêmes : contrôle total de ce qui est tracé
print("MLflow :", TRACKING_URI.split("@")[-1], "| expérience :", EXPERIMENT)

## 3. Entraînement + évaluation
- Mêmes données, epochs, batch, imgsz et seed pour les deux modèles ; pas d'early stopping (`patience: 0`).
- `best.pt` est choisi par Ultralytics sur la **validation source** ; les tests ne servent qu'à mesurer.
- `F1_macro` = F1 moyen par classe au seuil de confiance qui le maximise (définition Ultralytics). Ce n'est **pas encore**
  le F1 du challenge CRDDC : il sera recalculé, avec intervalles de confiance bootstrap, à partir des `predictions.json`
  sauvegardés ici (pas besoin de réentraîner).

In [ ]:
RUNS = Path("/kaggle/working/runs")
SOURCE = sorted(c for c, v in CFG["data"]["countries"].items() if v["role"] == "source")
PRESENT = sorted({k.split("/")[0] for k in STATS["per_country_split"]})


def clean(name):
    return re.sub(r"[^\w\-./ ]", "_", name)  # caractères acceptés par MLflow


def box_metrics(m, prefix):
    b = m.box
    out = {f"{prefix}/mAP50": float(b.map50), f"{prefix}/mAP50-95": float(b.map),
           f"{prefix}/precision": float(b.mp), f"{prefix}/recall": float(b.mr),
           f"{prefix}/F1_macro": float(np.mean(b.f1)) if len(b.f1) else float("nan")}
    for i, c in enumerate(b.ap_class_index):
        out[f"{prefix}/AP50_{CLASSES[int(c)]}"] = float(b.ap50[i])
        out[f"{prefix}/AP50-95_{CLASSES[int(c)]}"] = float(b.ap[i])
    return out


def evaluate(cls, weights, data_yaml, split, tag, run_dir):
    m = cls(str(weights)).val(data=str(data_yaml), split=split, imgsz=TRAIN["imgsz"], batch=TRAIN["batch"],
                              conf=CFG["eval"]["conf_map"], save_json=True, plots=True, device=0,
                              project=str(run_dir), name=f"eval_{tag}", exist_ok=True, verbose=False)
    met = box_metrics(m, tag)
    met[f"{tag}/gpu_ms_per_img"] = float(m.speed["inference"])
    mlflow.log_metrics({clean(k): v for k, v in met.items()})
    pred = Path(m.save_dir) / "predictions.json"
    if pred.exists():
        mlflow.log_artifact(str(pred), artifact_path=f"predictions/{tag}")
    return met


def run_baseline(key):
    spec = TRAIN["models"][key]
    cls = YOLO if spec["arch"] == "yolo" else RTDETR
    epochs, fraction = (1, 0.05) if SMOKE else (TRAIN["epochs"], 1.0)
    run_name = run_name_for(key)
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({"stage": "S1-baseline", "smoke": SMOKE, "git_sha": GIT_SHA,
                         "dataset_sha256": DATASET_INFO["manifest_sha256"], **VERSIONS})
        mlflow.log_params({"model": key, "weights": spec["weights"], "epochs": epochs, "fraction": fraction,
                           "imgsz": TRAIN["imgsz"], "batch": TRAIN["batch"], "seed": SEED,
                           "patience": TRAIN["patience"], "deterministic": TRAIN["deterministic"],
                           "amp": TRAIN["amp"], "train_countries": ",".join(SOURCE),
                           "optimizer": spec.get("optimizer", "auto"), "lr0": spec.get("lr0", "auto")})
        mlflow.log_artifact(f"{REPO_DIR}/configs/default.yaml", artifact_path="config")
        for f in ("dataset_info.json", "stats.json", "manifest.csv"):
            mlflow.log_artifact(str(DATA / f), artifact_path="dataset")

        model = cls(spec["weights"])
        optim = {k: spec[k] for k in ("optimizer", "lr0") if k in spec}  # recette propre à l'architecture
        t0 = time.time()
        model.train(**optim, data=str(DATA / "data_source.yaml"), epochs=epochs, fraction=fraction, imgsz=TRAIN["imgsz"],
                    batch=TRAIN["batch"], seed=SEED, deterministic=TRAIN["deterministic"],
                    patience=TRAIN["patience"], workers=TRAIN["workers"], amp=TRAIN["amp"], device=0,
                    project=str(RUNS), name=run_name, exist_ok=True, plots=True)
        train_min = (time.time() - t0) / 60
        save_dir = Path(model.trainer.save_dir)
        mlflow.log_metric("time/train_minutes", train_min)
        mlflow.log_metric("gpu/max_mem_GB", torch.cuda.max_memory_allocated() / 1e9)

        df = pd.read_csv(save_dir / "results.csv")
        df.columns = [c.strip() for c in df.columns]
        for _, row in df.iterrows():
            mlflow.log_metrics({f"epoch/{clean(c)}": float(row[c]) for c in df.columns
                                if c != "epoch" and pd.notna(row[c])}, step=int(row["epoch"]))

        best = save_dir / "weights" / "best.pt"
        info = {}
        try:  # mesuré sur le modèle entraîné (tête à 4 classes), pas sur les poids COCO
            _, n_params, _, gflops = cls(str(best)).info(verbose=True)  # verbose=False renvoie None (8.4)
            info = {"model/params_M": n_params / 1e6, "model/GFLOPs": float(gflops)}
            mlflow.log_metrics(info)
        except Exception as e:  # l'API info() change selon les versions
            print("model.info indisponible :", e)
        row = {"model": key, "run_id": run.info.run_id, "train_min": round(train_min, 1),
               **{k.split("/")[1]: v for k, v in info.items()}}
        row.update(evaluate(cls, best, DATA / "data_source.yaml", "val", "source_val", save_dir))
        row.update(evaluate(cls, best, DATA / "data_source.yaml", "test", "source_test", save_dir))
        for c in PRESENT:
            row.update(evaluate(cls, best, DATA / f"data_eval_{c}.yaml", "val", f"test_{c}", save_dir))
            # Contrôle à conditions égales : images avec >= 1 dégât (le test États-Unis n'a aucune route saine).
            row.update(evaluate(cls, best, DATA / f"data_eval_{c}_pos.yaml", "val", f"test_{c}_pos", save_dir))
        mlflow.log_artifacts(str(save_dir), artifact_path="ultralytics_run")  # poids, courbes, matrices de confusion
        return row

In [ ]:
rows = []
for key in MODELS_TO_RUN:
    rows.append(run_baseline(key))
    torch.cuda.empty_cache()

res = pd.DataFrame(rows)
out_dir = Path("/kaggle/working/results"); out_dir.mkdir(exist_ok=True)
out_csv = out_dir / f"baselines_{'-'.join(run_name_for(k).removeprefix('baseline_') for k in MODELS_TO_RUN)}.csv"
res.to_csv(out_csv, index=False)

summary_cols = ["model", "params_M", "GFLOPs", "train_min",
                "source_test/mAP50", "source_test/mAP50-95", "source_test/F1_macro"] + \
               [f"test_{c}{v}/mAP50" for c in PRESENT for v in ("", "_pos")]
display(res[[c for c in summary_cols if c in res.columns]].round(3).T)
print("CSV complet (toutes les métriques, par classe et par pays) :", out_csv)

## 3b. Intervalles de confiance (bootstrap, 1000 tirages d'images)
mAP@50 au sens COCO (le module documente l'écart avec la formule d'Ultralytics, gardée pour contrôle dans
`mAP50_ultralytics_method`) et F1 micro « CRDDC » au seuil de confiance choisi sur la validation source.
Tourne sur CPU en quelques minutes ; les CSV sont aussi attachés au run MLflow.

In [ ]:
for key in MODELS_TO_RUN:
    run_name = run_name_for(key)
    ci_csv = out_dir / f"ci_{run_name.removeprefix('baseline_')}.csv"
    !python -m roadshift.eval.report --runs-dir {RUNS / run_name} --data {DATA_OUT} --model {key} --out {ci_csv}
    ci = pd.read_csv(ci_csv)
    with mlflow.start_run(run_id=res.loc[res.model == key, "run_id"].iloc[0]):
        mlflow.log_artifact(str(ci_csv), artifact_path="results")
        mlflow.log_param("conf_F1_from_source_val", float(ci.conf_F1.iloc[0]))
        for _, r in ci.iterrows():
            mlflow.log_metrics({f"ci/{r['eval']}/{k}": float(r[k])
                                for k in ("mAP50", "mAP50_lo", "mAP50_hi", "F1", "F1_lo", "F1_hi")})
    display(ci[["eval", "n_images", "n_boxes", "mAP50", "mAP50_lo", "mAP50_hi", "F1", "F1_lo", "F1_hi"]])

## 4. Récupérer les sorties (onglet *Output* de la version, ou téléchargement direct)

In [ ]:
import shutil

meta = Path("/kaggle/working/dataset_meta"); meta.mkdir(exist_ok=True)
for f in ["dataset_info.json", "stats.json", "manifest.csv", *[p.name for p in DATA.glob("list_*.txt")]]:
    (meta / f).write_bytes((DATA / f).read_bytes())  # splits exacts : à versionner dans le dépôt
shutil.copytree(DATA / "labels", meta / "labels", dirs_exist_ok=True)  # vérité terrain : recalcul des IC hors Kaggle
!cd /kaggle/working && zip -qr s1_outputs.zip results dataset_meta mlflow.db mlruns 2>/dev/null; ls -lh /kaggle/working

## À me renvoyer
1. Après le passage `SMOKE = True` : le tableau de la section 1 (images/boîtes par pays et split), `unknown_labels`,
   `dropped_boxes`, et la durée `train_min` + `gpu/max_mem_GB` du smoke run (pour calibrer le budget du vrai run).
2. Après chaque run complet : le CSV `results/baselines_<modèle>.csv`.

Aucune conclusion n'est tirée avant ces chiffres.